# RAG (Retrieval Augmented Generation) - Complete Tutorial

This notebook demonstrates how to use your vector database for RAG:
1. Retrieve relevant context from your embeddings
2. Generate answers using an LLM
3. Get answers with source citations

## What is RAG?

**RAG = Retrieval Augmented Generation**

Instead of asking an LLM questions directly, we:
1. Find relevant information from YOUR documents (retrieval)
2. Give that information to the LLM as context
3. LLM answers based on YOUR data (generation)

**Benefits:**
- LLM answers from YOUR documents (not general knowledge)
- Reduces hallucinations
- Provides source citations
- Works with private/recent data

## Setup and Imports

In [1]:
import sys
sys.path.append('..')

from vectordb.embeddings import create_embed_function, EmbeddingConfig
from vectordb.vector_store import get_backend, VectorStoreConfig
from vectordb.rag import (
    rag_pipeline,
    rag_query,
    get_llm_provider,
    RAGConfig,
    format_response_with_sources,
    retrieve_context,
    format_context,
    build_prompt
)

## Step 1: Connect to Your Vector Database

First, connect to your existing vector database with embeddings.

In [2]:
# Configure your vector store (adjust to match your setup)
vector_config = VectorStoreConfig(
    collection_name="my_documents",  # Your collection name
    distance_metric="cosine"
)

# Connect to backend (use "simple" for JSON-based or "pgvector" for PostgreSQL)
backend = get_backend("pgvector", vector_config)

print(f"Connected to vector database")
print(f"Total embeddings: {backend.count()}")

Connected to vector database
Total embeddings: 750


## Step 2: Create Embedding Function

This converts your queries into vectors (same model as your stored embeddings).

In [3]:
# Create embedding function (use the SAME model as your stored embeddings!)
embedding_config = EmbeddingConfig(
    model="all-MiniLM-L6-v2",  # Match your embedding model
    normalize=True
)

embed_fn = create_embed_function("sentence-transformers", embedding_config)

print("Embedding function ready")

Embedding function ready


## Step 3: Choose Your LLM Provider

You have several options:

### Option A: OpenAI (Cloud, requires API key)
```python
provider_type = "openai"
rag_config = RAGConfig(
    model="gpt-3.5-turbo",  # or "gpt-4", "gpt-4-turbo"
    api_key="your-api-key"  # or set OPENAI_API_KEY env var
)
```

### Option B: Anthropic Claude (Cloud, requires API key)
```python
provider_type = "anthropic"
rag_config = RAGConfig(
    model="claude-3-5-sonnet-20241022",
    api_key="your-api-key"  # or set ANTHROPIC_API_KEY env var
)
```

### Option C: Ollama (Local, FREE!)
```python
# First install Ollama: https://ollama.ai/
# Then: ollama pull llama3.2

provider_type = "ollama"
rag_config = RAGConfig(
    model="llama3.2",  # or "mistral", "phi", "gemma", etc.
)
```

### Option D: Groq (Cloud, FREE tier available)
```python
provider_type = "groq"
rag_config = RAGConfig(
    model="llama-3.1-70b-versatile",
    api_key="your-groq-api-key"  # or set GROQ_API_KEY env var
)
```

In [8]:
# CHOOSE YOUR PROVIDER HERE

# Example: Using OpenAI (uncomment and add your API key)
# provider_type = "openai"
# rag_config = RAGConfig(
#     model="gpt-3.5-turbo",
#     temperature=0.7,
#     max_tokens=1000,
#     top_k=5,  # Number of chunks to retrieve
# )

# Example: Using Ollama (local, free)
provider_type = "ollama"
rag_config = RAGConfig(
    model="qwen3-vl:8b",
    temperature=0.7,
    max_tokens=1000,
    top_k=5,
)

print(f"Using provider: {provider_type}")
print(f"Model: {rag_config.model}")

Using provider: ollama
Model: qwen3-vl:8b


## Step 4: Ask Questions - Simple Method

Use `rag_pipeline()` for the easiest way to get answers.

In [9]:
# Ask a question
question = "What is the main topic of the documents?"

response = rag_pipeline(
    question=question,
    backend=backend,
    embed_fn=embed_fn,
    provider_type=provider_type,
    config=rag_config
)

# Display formatted response with sources
print(format_response_with_sources(response))

ANSWER:
Based on the context provided, the main topic of the documents is Three.js programming for creating 3D graphics and visualizations. The context shows multiple chapters (specifically Chapter 5) that cover various aspects of working with Three.js:

- Visualizing audio data with particle systems (Source 1)
- Using Blender for 3D modeling integration (Source 2)
- Creating parametric geometries like trees using libraries (Source 3)
- Programmatic geometries creation (Source 4)
- Skeletal-based animation techniques (Source 5)

The documents collectively focus on different techniques for creating 3D content using the Three.js library, including both standard geometries and custom geometry creation methods. The context specifically mentions "Chapter 5" across multiple sources, indicating these are excerpts from a book or tutorial about Three.js development.

SOURCES:

[1] clean.md (Relevance: 0.061)
    ## Visualizing Audio Data with a Particle System
## [ 86 ]
}
## // add the new to t

## Step 5: Understanding the RAG Process Step-by-Step

Let's break down what happens behind the scenes:

In [10]:
question = "Explain the key concepts from the documents"

print("="*80)
print("STEP-BY-STEP RAG PROCESS")
print("="*80)

# STEP 1: Retrieve relevant chunks
print("\n[STEP 1] Retrieving relevant chunks from vector database...")
search_results = retrieve_context(
    query=question,
    backend=backend,
    embed_fn=embed_fn,
    top_k=rag_config.top_k
)

print(f"Found {len(search_results)} relevant chunks:")
for i, result in enumerate(search_results, 1):
    print(f"  {i}. {result.chunk.metadata.get('file_name', result.chunk.source_file)} (score: {result.score:.3f})")

# STEP 2: Format context
print("\n[STEP 2] Formatting context for LLM...")
context = format_context(search_results, rag_config.context_max_length)
print(f"Context length: {len(context)} characters")
print(f"\nContext preview:\n{context[:500]}...\n")

# STEP 3: Build prompt
print("[STEP 3] Building prompt...")
prompt = build_prompt(question, context)
print(f"Prompt length: {len(prompt)} characters")
print(f"\nPrompt preview:\n{prompt[:500]}...\n")

# STEP 4: Generate answer
print("[STEP 4] Generating answer with LLM...")
provider = get_llm_provider(provider_type, rag_config)
answer, tokens = provider.generate(prompt, rag_config.system_prompt)

print(f"\n{'='*80}")
print("FINAL ANSWER:")
print(f"{'='*80}")
print(answer)
print(f"\nTokens used: {tokens}" if tokens else "")

STEP-BY-STEP RAG PROCESS

[STEP 1] Retrieving relevant chunks from vector database...
Found 3 relevant chunks:
  1. clean.md (score: 0.148)
  2. clean.md (score: 0.124)
  3. clean.md (score: 0.054)

[STEP 2] Formatting context for LLM...
Context length: 1281 characters

Context preview:
[Source 1: clean.md (Score: 0.148)]
remote: Counting objects: 36, done.
remote: Compressing objects: 100% (21/21), done.
## remote: Total 44 (delta 6), reused 0 (delta 0)
Unpacking objects: 100% (44/44), done.
## Checking connectivity... done

---
[Source 2: clean.md (Score: 0.124)]
# Chapter 4
## [ 77 ]
## analyser2 = context.createAnalyser();
## analyser2.smoothingTimeConstant = 0.4;
## analyser2.fftSize = 1024;
## // connect them together
## sourceNode.connect(splitter);
## splitter.connect(a...

[STEP 3] Building prompt...
Prompt length: 1530 characters

Prompt preview:
Answer the question based on the context provided below. If the context doesn't contain enough information to answer the question, 

## Step 6: Multiple Questions

Ask several questions at once:

In [ ]:
questions = [
    "What are the main topics covered?",
    "What technical concepts are explained?",
    "Are there any code examples?"
]

for i, question in enumerate(questions, 1):
    print(f"\n{'='*80}")
    print(f"QUESTION {i}: {question}")
    print(f"{'='*80}")
    
    response = rag_pipeline(
        question=question,
        backend=backend,
        embed_fn=embed_fn,
        provider_type=provider_type,
        config=rag_config
    )
    
    print(f"\nAnswer:\n{response.answer}")
    print(f"\nSources: {len(response.sources)} chunks")
    for j, src in enumerate(response.sources[:3], 1):
        file_name = src.chunk.metadata.get('file_name', src.chunk.source_file)
        print(f"  {j}. {file_name} (relevance: {src.score:.3f})")

## Step 7: Customizing RAG Behavior

You can customize various aspects of the RAG pipeline:

In [ ]:
# Custom configuration
custom_config = RAGConfig(
    model="gpt-3.5-turbo",  # or your chosen model
    temperature=0.3,  # Lower = more focused, Higher = more creative
    max_tokens=500,   # Shorter responses
    top_k=10,         # Retrieve more chunks
    context_max_length=6000,  # More context
    include_sources=True,
    system_prompt="You are a helpful assistant that answers questions based solely on the provided context. Be concise and cite sources."
)

# Use custom config
response = rag_pipeline(
    question="Summarize the key points",
    backend=backend,
    embed_fn=embed_fn,
    provider_type=provider_type,
    config=custom_config
)

print(response.answer)

## Step 8: Inspecting Retrieved Context

Sometimes you want to see what chunks were retrieved before generating an answer:

In [ ]:
question = "What are the important details?"

# Retrieve without generating
search_results = retrieve_context(question, backend, embed_fn, top_k=5)

print(f"Retrieved {len(search_results)} chunks:\n")

for i, result in enumerate(search_results, 1):
    print(f"{'='*80}")
    print(f"Chunk {i} (Relevance: {result.score:.3f})")
    print(f"{'='*80}")
    print(f"Source: {result.chunk.metadata.get('file_name', result.chunk.source_file)}")
    print(f"Content:\n{result.chunk.content}\n")

## Step 9: Advanced - Manual RAG Pipeline

For full control, you can build the pipeline manually:

In [ ]:
from vectordb.rag import generate_answer

# Manual pipeline
question = "What should I know about this topic?"

# 1. Retrieve
results = retrieve_context(question, backend, embed_fn, top_k=3)

# 2. Format context (you can customize this)
custom_context = "\n\n".join([
    f"Document {i}: {r.chunk.content}"
    for i, r in enumerate(results, 1)
])

# 3. Create custom prompt
custom_prompt = f"""
Based on these documents, answer the question.

Documents:
{custom_context}

Question: {question}

Provide a detailed answer:
"""

# 4. Generate
provider = get_llm_provider(provider_type, rag_config)
answer, tokens = provider.generate(custom_prompt)

print(answer)

## Step 10: Comparing With and Without RAG

See the difference between RAG (with your documents) and direct LLM queries:

## Summary

### What You've Learned:

✅ **How RAG works**: Retrieval → Context → Generation  
✅ **Simple usage**: `rag_pipeline()` for easy queries  
✅ **Step-by-step process**: Understanding each stage  
✅ **Multiple LLM providers**: OpenAI, Anthropic, Ollama, Groq  
✅ **Customization**: Adjusting retrieval and generation parameters  
✅ **Source citations**: Tracking where answers come from  

### Key Functions:

```python
# Simple - All in one
response = rag_pipeline(question, backend, embed_fn, provider_type, config)

# Advanced - Step by step
results = retrieve_context(question, backend, embed_fn, top_k)
context = format_context(results)
prompt = build_prompt(question, context)
answer, tokens = provider.generate(prompt)
```

### Next Steps:

- Experiment with different models and parameters
- Try different question types
- Adjust `top_k` to retrieve more/fewer chunks
- Customize prompts for your use case
- Add conversation history for chat-like interactions
- Build a complete application (web interface, API, etc.)

## Configuration Tips

### Choosing `top_k` (number of chunks):
- **top_k=3**: Fast, focused answers
- **top_k=5**: Balanced (recommended)
- **top_k=10**: Comprehensive, but may include less relevant info

### Choosing `temperature`:
- **0.0-0.3**: Focused, deterministic, factual
- **0.4-0.7**: Balanced (recommended)
- **0.8-1.0**: Creative, varied responses

### Choosing `max_tokens`:
- **100-300**: Brief answers
- **500-1000**: Standard (recommended)
- **1500+**: Detailed, comprehensive answers

### LLM Provider Recommendations:
- **OpenAI GPT-4**: Best quality, expensive
- **OpenAI GPT-3.5-turbo**: Good quality, affordable
- **Claude 3.5 Sonnet**: Excellent quality, good for long context
- **Ollama (Llama 3.2)**: FREE, runs locally, good quality
- **Groq**: Very fast, FREE tier available